In [ ]:
import Modules.Plan as P, inspect
print(P.__file__)
print(inspect.getsource(P._InterpAdjoint.backward))

In [ ]:
%matplotlib widget

import sys
import time
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import itertools
import numpy as np
import pypulseq as pp
from pypulseq import Sequence
import torch
import torch.nn as nn
import torchkbnufft as tkbn
from Modules.CoilSens import cyclic_head_coil
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import ChannelOperator, LinearOperator
from Modules.NumSolvers import conjugate_gradient
from Modules.Regularizer import Regularizer
from Modules.fast_iso import isochromat_sim_fast
from Modules.Plan import KbNufft, KbNufftAdjoint
from tqdm import tqdm

sys.path.insert(0, "..")
from Modules import default_system, load_brain_phantom

torch.set_default_device("cuda")
device = torch.device("cuda")

In [ ]:
def main(
    test_report = False,
    FOV = 250e-3,
    N_x = 64,  
    alpha = 5,  
    slice_thickness = 3e-3,  
    TE = 3.98e-3, 
    TR = 100e-3,  
    N_spokes = 8000, 
    N_dummy = 20,  
    rf_spoiling_inc = 117, 
    write_seq: bool = False,
) -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / FOV

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=2e-3,
        flip_angle=alpha * np.pi / 180,
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=4,
        return_gz=True,
    )

    gx_base = pp.make_trapezoid(
        channel="x", flat_area=N_x * deltak, flat_time=6.4e-3 / 5, system=system
    )
    adc = pp.make_adc(
        num_samples=N_x, duration=gx_base.flat_time, delay=gx_base.rise_time, system=system
    )
    gx_pre_base = pp.make_trapezoid(
        channel="x", area=-gx_base.area / 2, duration=2e-3, system=system
    )
    gz_reph = pp.make_trapezoid(
        channel="z", area=-gz.area / 2, duration=2e-3, system=system
    )
    gx_spoil_base = pp.make_trapezoid(channel="x", area=0.5 * N_x * deltak, system=system)
    gz_spoil = pp.make_trapezoid(channel="z", area=4 / slice_thickness, system=system)

    delay_TE = (
        np.ceil(
            (
                TE
                - pp.calc_duration(gx_pre_base)
                - gz.fall_time
                - gz.flat_time / 2
                - pp.calc_duration(gx_base) / 2
            )
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )
    delay_TR = (
        np.ceil(
            (
                TR
                - pp.calc_duration(gx_pre_base)
                - pp.calc_duration(gz)
                - pp.calc_duration(gx_base)
                - delay_TE
            )
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )
    assert np.all(delay_TR) > pp.calc_duration(gx_spoil_base, gz_spoil)
    rf_phase = 0
    rf_inc = 0

    for i in range(N_spokes + N_dummy):

        
        rf.phase_offset = rf_phase / 180 * np.pi
        adc.phase_offset = rf_phase / 180 * np.pi

        rf_inc = (rf_inc + rf_spoiling_inc) % 360
        rf_phase = (rf_phase + rf_inc) % 360

        spoke_idx = i - N_dummy
        if spoke_idx >= 0:
            phi = np.arccos(1 - 2 * (spoke_idx + 0.5) / N_spokes)
            theta = np.pi * (1 + np.sqrt(5)) * spoke_idx

            kx = np.sin(phi) * np.cos(theta)
            ky = np.sin(phi) * np.sin(theta)
            kz = np.cos(phi)
        else:
            
            kx, ky, kz = 1, 0, 0

        
        seq.add_block(rf, gz)

        
        gx_pre = pp.make_trapezoid(
            channel="x", area=gx_pre_base.area * kx, duration=2e-3, system=system
        )

        gy_pre = pp.make_trapezoid(
            channel="y", area=gx_pre_base.area * ky, duration=2e-3, system=system
        )

        gz_pre_total = pp.make_trapezoid(
            channel="z",
            area=gx_pre_base.area * kz + gz_reph.area,
            duration=2e-3,
            system=system,
        )

        seq.add_block(gx_pre, gy_pre, gz_pre_total)
        seq.add_block(pp.make_delay(delay_TE))

        gx = pp.make_trapezoid(
            channel="x",
            amplitude=gx_base.amplitude * kx,
            flat_time=gx_base.flat_time,
            rise_time=gx_base.rise_time,
            fall_time=gx_base.fall_time,
            system=system,
        )
        gy = pp.make_trapezoid(
            channel="y",
            amplitude=gx_base.amplitude * ky,
            flat_time=gx_base.flat_time,
            rise_time=gx_base.rise_time,
            fall_time=gx_base.fall_time,
            system=system,
        )
        gz_ro = pp.make_trapezoid(
            channel="z",
            amplitude=gx_base.amplitude * kz,
            flat_time=gx_base.flat_time,
            rise_time=gx_base.rise_time,
            fall_time=gx_base.fall_time,
            system=system,
        )

        if spoke_idx >= 0:
            seq.add_block(gx, gy, gz_ro, adc)
        else:
            seq.add_block(gx, gy, gz_ro)

        gx_spoil = pp.make_trapezoid(
            channel="x", area=gx_spoil_base.area * kx, system=system
        )
        gy_spoil = pp.make_trapezoid(
            channel="y", area=gx_spoil_base.area * ky, system=system
        )
        gz_spoil_total = pp.make_trapezoid(
            channel="z",
            area=gx_spoil_base.area * kz + gz_spoil.area,
            system=system,
        )

        seq.add_block(
            gx_spoil,
            gy_spoil,
            gz_spoil_total,
            pp.make_delay(delay_TR),
        )

    ok, error_report = seq.check_timing()
    if ok:
        print("Timing check passed successfully")
    else:
        print("Timing check failed! Error listing follows:")

    if write_seq:
        seq.set_definition("FOV", [FOV, FOV, slice_thickness])
        seq.set_definition("Name", "gre_rad")
        seq.write("3D Kooshball GRE.seq")


if __name__ == "__main__":
    main(write_seq=True)

In [ ]:
device = torch.device("cuda")

# ---- single source of truth ------------------------------------------------
FOV        = 250e-3
n          = 128      # phantom + recon matrix (n^3)
N_x        = n         # readout samples/spoke -> kmax matches the recon grid (was 320 vs a 64 grid)
N_spokes   = 5700    # ~ (pi/2)*n^2 spokes for full sampling
N_channels = 8
spin_count = 64       # isochromats per voxel (2D notebook used 256)

main(FOV=FOV, N_x=N_x, N_spokes=N_spokes, write_seq=True)
mr_seq = mr0.Sequence.import_file("3D Kooshball GRE.seq")

# ---- phantom (BrainWeb) ----------------------------------------------------
phantom = load_brain_phantom(
    raw_path="subject04_crisp_v.rawb",
    target_size=(n, n, n),
    device=device,
    permute_order=(2, 1, 0),
)
print(type(phantom).__name__, "size =", phantom.size)
#assert torch.allclose(phantom.size.cpu().float(), torch.full((3,), FOV), atol=1e-4), (
    #"phantom.size must equal the FOV cube so that the phantom grid == the recon grid "
    #"(pad the BrainWeb volume to a cube before resampling)."
#)

# ---- coils (same call as the 2D notebook, now with a real 3rd axis) ---------
scale = tuple(float(s) / FOV for s in phantom.size)          # == (1,1,1) here
coil_sim = cyclic_head_coil((N_channels, n, n, n), fov_scaling=scale).to(torch.complex64)
phantom.coil_sens = coil_sim.to(device)
data = phantom.build()

# ---- isochromat simulation -------------------------------------------------
torch.manual_seed(0)
signal = isochromat_sim_fast(
    mr_seq, data.cuda(), spin_count=spin_count, spin_dist="rand", print_progress=True,
)
signal = signal.permute((1, 0)).contiguous().to(torch.complex64)   # (N_channels, N_adc)
print("simulated signal:", tuple(signal.shape))
torch.save({"data": data.cpu(), "k": k_traj_adc.cpu(), "coil": coil_sim.cpu()},
           "recon_inputs.pt")

In [ ]:
def kspace_to_rad(mr_seq, n, FOV):
    """ADC k-space (1/m) -> rad/pixel of an n^3 grid spanning `FOV`. (3, N_adc)"""
    k = torch.as_tensor(mr_seq.get_kspace()).float()
    k = k.reshape(-1, k.shape[-1])[:, :3].T.contiguous()
    return k * (2 * np.pi * FOV / n)


def nufft_recon(signal, k_rad, n):
    """Per-channel density-compensated adjoint NUFFT. signal: (C, N) -> (C, n, n, n)"""
    k_rad = k_rad.to(signal.device)
    keep = k_rad.norm(dim=0) <= np.pi                 # drop (don't clamp!) samples beyond the grid's kmax
    k, s = k_rad[:, keep], signal[:, keep]
    print(f"using {int(keep.sum())}/{keep.numel()} samples")
    dcomp = k.norm(dim=0).clamp(min=np.pi / n) ** 2   # |k|^2 for radial
    adj = tkbn.KbNufftAdjoint(im_size=(n, n, n), grid_size=(2 * n,) * 3).to(signal.device)
    return adj((s * dcomp)[None], k)[0]


def _rel_profile(a):
    a = np.abs(a)
    return a / (np.sqrt((a ** 2).sum(0, keepdims=True)) + 1e-12)


def match_coil_orientation(coil, img, mask_frac=0.3):
    """Which of the 48 axis permutations/flips of `coil` (C,n,n,n) matches the orientation of the
    raw NUFFT images `img` (C,n,n,n)?  Compares |coil_c|/rss(coil) with |img_c|/rss(img) inside the head."""
    coil, img = np.asarray(coil), np.asarray(img)
    rss = np.sqrt((np.abs(img) ** 2).sum(0))
    mask = rss > mask_frac * rss.max()
    ref = _rel_profile(img)[:, mask].ravel(); ref = ref - ref.mean()
    best = (-2.0, None, None)
    for perm in itertools.permutations((0, 1, 2)):
        cp = coil.transpose(0, *[p + 1 for p in perm])
        for flips in itertools.product([False, True], repeat=3):
            dims = tuple(d + 1 for d, f in enumerate(flips) if f)
            cand = np.flip(cp, dims) if dims else cp
            v = _rel_profile(cand)[:, mask].ravel(); v = v - v.mean()
            score = float(ref @ v / (np.linalg.norm(ref) * np.linalg.norm(v) + 1e-12))
            if score > best[0]:
                best = (score, perm, flips)
    score, perm, flips = best
    dims = tuple(d + 1 for d, f in enumerate(flips) if f)
    out = coil.transpose(0, *[p + 1 for p in perm])
    return np.ascontiguousarray(np.flip(out, dims) if dims else out), perm, flips, score

In [ ]:
k_rad = kspace_to_rad(mr_seq, n, FOV)
assert k_rad.shape[1] == signal.shape[1], (k_rad.shape, signal.shape)   # ADC order must match

img = nufft_recon(signal, k_rad, n)                                     # (C, n, n, n), raw NUFFT orientation

coil_np, perm, flips, score = match_coil_orientation(coil_sim.cpu().numpy(), img.cpu().numpy())
print(f"coil orientation: perm={perm} flips={flips} score={score:.3f}")   # want score close to 1
coil_recon = torch.from_numpy(coil_np).to(img.device)

den = coil_recon.abs().square().sum(0)
combined = (coil_recon.conj() * img).sum(0) / den.clamp(min=1e-2 * den.max())   # sensitivity-weighted combination
rss = img.abs().square().sum(0).sqrt()

mid = n // 2
def show(ax, vol, title):
    ax.imshow(torch.rot90(vol.abs(), k=1, dims=(0, 1))[:, :, mid].cpu().numpy(), cmap="gray")
    ax.set_title(title); ax.axis("off")

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
show(ax[0], rss, "RSS (no coil maps)")
show(ax[1], combined, "sens-weighted combination")
show(ax[2], img[0], "channel 0 alone")
plt.show()

In [ ]:
seq = pp.Sequence()
seq.read("3D Kooshball GRE.seq")
data = signal.clone()
data = data.contiguous()
kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:3, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

In [ ]:
from torch.utils.checkpoint import checkpoint

class DeepADMM(nn.Module):
    def __init__(self, shape, rho_init=0.1, maxiter_admm=10, maxiter_cg=5,
                 kernel_size=(3, 3, 3), features=16, num_of_resblocks=4,
                 scale_factor=1, checkpoint_iterations=True):
        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(
            shape, kernel_size=kernel_size, features=features,
            num_of_resblocks=num_of_resblocks, scale_factor=scale_factor,
        )
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg
        self.checkpoint_iterations = checkpoint_iterations

    def _step(self, x, z, u, Q, b):
        x = conjugate_gradient(Q + self.rho * self.I, b=b + self.rho * (z - u),
                               x=x, maxiter=self.maxiter_cg, disable_progressbar=True)
        z = self.regularizer(x + u)
        u = u + x - z
        return x, z, u

    def forward(self, Q, b, x, disable_progressbar=True):
        z = torch.zeros(self.I.shape[0], dtype=x.dtype)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)
        use_ckpt = self.checkpoint_iterations and self.training and torch.is_grad_enabled()
        for _ in range(self.maxiter_admm):
            if use_ckpt:
                x, z, u = checkpoint(self._step, x, z, u, Q, b, use_reentrant=False)
            else:
                x, z, u = self._step(x, z, u, Q, b)
        return x

In [ ]:
class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities

        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class NonuniformFourierTransformOperator(LinearOperator):
    """F: image (nX,nY,nZ) -> multi-coil k-space (nC, nK). k: (3, nK) in cycles/voxel."""

    def __init__(self, k, input_shape, device=None):
        nD, nK = k.shape                      # (3, nK)
        nC, nX, nY, nZ = input_shape
        self.n_modes = (nX, nY, nZ)
        self.k = (-k).unsqueeze(-1)           # (3, nK, 1), sign flipped as in the Kooshball notebooks
        self.forward_shape = (nC, nX, nY, nZ, 1)
        self.adjoint_shape = (nC, nK)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)
        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

class GradientOperator(LinearOperator):

    def __init__(
        self,
        dim: int,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        self.dim = dim
        self.forward_shape = data_shape
        self.adjoint_shape = data_shape
        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = x - torch.roll(x, 1, self.dim)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = x - torch.roll(x, -1, self.dim)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class IdentityOperator(LinearOperator):

    def __init__(
        self,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):

        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        return x

In [ ]:
F = NonuniformFourierTransformOperator(k_traj_adc, (N_channels, N_x, N_x, N_x))
C = ChannelOperator(coil_sim, (N_x, N_x, N_x))
A = F @ C

In [ ]:
AHd = A.H @ data.view(torch.float32).ravel()
AHd_image = AHd.view(torch.complex64).reshape((N_x, N_x, N_x))
Q = A.H @ A
b = AHd
x = torch.zeros(N_x * N_x * N_x, dtype=torch.complex64).view(torch.float32)
x_recon = conjugate_gradient(Q, b, x, maxiter=30)
x_recon_image = x_recon.view(torch.complex64).reshape((N_x, N_x, N_x))

In [ ]:
vol = torch.abs(x_recon_image).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Slices through the center of the volume with a 180-degree rotation (2 full 90-degree turns)
axes[0].imshow(np.rot90(vol[N_x // 2, :, :].T, 2), cmap="gray")
axes[0].set_title("Plane 1 (X-slice)")

axes[1].imshow(np.rot90(vol[:, N_x // 2, :].T, 2), cmap="gray")
axes[1].set_title("Plane 2 (Y-slice)")

axes[2].imshow(np.rot90(vol[:, :, N_x // 2].T, 2), cmap="gray")
axes[2].set_title("Plane 3 (Z-slice)")

for ax in axes:
    ax.axis("off")

plt.suptitle("CG Reconstruction")
plt.show()

In [ ]:
torch.manual_seed(0)
shape = (N_x, N_x, N_x, 1)
model_init = DeepADMM(shape, rho_init=0.1, maxiter_admm=4, maxiter_cg=25)

with torch.no_grad():
    x_deep_admm_init = model_init(Q, b, b)
x_deep_admm_init_image = x_deep_admm_init.view(torch.complex64).reshape((N_x, N_x, N_x))

In [ ]:
import Modules.Plan
shape = (N_x, N_x, N_x, 1)

torch.manual_seed(0)
model = DeepADMM(
    shape,
    rho_init=0.1,
    maxiter_admm=4,
    maxiter_cg=25,
)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

nK = k_traj_adc.shape[1]
num_splits = 10

model.train()
losses = []

progress_bar = tqdm(range(num_splits), desc="Self-supervised splits")

for split in progress_bar:

    np.random.seed(split)

    indices = np.arange(nK)
    np.random.shuffle(indices)
    index_src, index_tar = np.split(indices, 2)

    k_traj_src = k_traj_adc[:, index_src]
    k_data_src = data[:, index_src].view(torch.float32).ravel()

    k_traj_tar = k_traj_adc[:, index_tar]
    k_data_tar = data[:, index_tar].view(torch.float32).ravel()

    F_src = NonuniformFourierTransformOperator(k_traj_src, (N_channels, N_x, N_x, N_x))
    F_tar = NonuniformFourierTransformOperator(k_traj_tar, (N_channels, N_x, N_x, N_x))
    C = ChannelOperator(coil_sim, (N_x, N_x, N_x))

    Q_src = C.H @ F_src.H @ F_src @ C
    b_src = C.H @ F_src.H @ k_data_src

    x_out = model(Q_src, b_src, b_src, disable_progressbar=True)

    k_data_pred = F_tar @ C @ x_out

    loss = torch.linalg.norm(k_data_pred - k_data_tar, ord=1) / torch.linalg.norm(
        k_data_tar, ord=1
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

print(torch.cuda.max_memory_allocated() / 1e9)
plt.figure()
plt.plot(losses, marker="o")
plt.xlabel("Split")
plt.ylabel("Self-supervised loss")
plt.title("DeepADMM training loss over splits")
plt.show()

In [ ]:
model.eval()

F_full = NonuniformFourierTransformOperator(k_traj_adc, (N_channels, N_x, N_x, N_x))
C = ChannelOperator(coil_sim, (N_x, N_x, N_x))

x0 = torch.zeros(N_x * N_x * N_x, dtype=torch.complex64).view(torch.float32)

with torch.no_grad():
    x_deep_admm_trained = model(Q, b, b)

x_deep_admm_trained_image = x_deep_admm_trained.view(torch.complex64).reshape(
    (N_x, N_x, N_x)
)

In [ ]:
vol = torch.abs(x_deep_admm_trained_image).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(np.rot90(vol[N_x // 2, :, :].T, 2), cmap="gray")
axes[0].set_title("Plane 1 (X-slice)")

axes[1].imshow(np.rot90(vol[:, N_x // 2, :].T, 2), cmap="gray")
axes[1].set_title("Plane 2 (Y-slice)")

axes[2].imshow(np.rot90(vol[:, :, N_x // 2].T, 2), cmap="gray")
axes[2].set_title("Plane 3 (Z-slice)")

for ax in axes:
    ax.axis("off")

plt.suptitle("DeepADMM (learned, trained) - Orthogonal Views")
plt.show()